# Map Coloring

This short notebook shows how *map coloring* can be formulated as a *constraint satisfaction problem*.
In map coloring, the goal is to color the states shown in a given map such that no two bordering states 
share the same color.  As an example, consider the map of Australia that is shown below.  Australia
has seven different states:
* [Western Australia](https://en.wikipedia.org/wiki/Western_Australia),
* [Northern Territory](https://en.wikipedia.org/wiki/Northern_Territory),
* [South Australia](https://en.wikipedia.org/wiki/South_Australia),
* [Queensland](https://en.wikipedia.org/wiki/Queensland),
* [New South Wales](https://en.wikipedia.org/wiki/New_South_Wales),
* [Victoria](https://en.wikipedia.org/wiki/Victoria_(Australia)), and
* [Tasmania](https://en.wikipedia.org/wiki/Tasmania).

As Tasmania is an island that does not share a border with any of the other states, it can be colored with any given color.

<img src="australia.png" alt="map of australia" width="500">

We use the backtracking constraint solver that has been developed in the notebook `02-Backtracking-Constraint-Solver.ipynb`.

In [ ]:
%load ../lib/RecursiveSet.jsh
%load ConstraintCompiler.jsh
%load 02-Backtracking-Constraint-Solver.jsh

The backtracking constraint solver requires numeric domains, so we map our colors to integers.

In [ ]:
Map<Integer, String> COLOR_MAP = Map.of(1, "red", 2, "blue", 3, "green");

The function $\texttt{mapColoringCSP}()$ returns a CSP that encodes the *map coloring problem* for Australia.

In [ ]:
CSP mapColoringCSP() {
    var variables   = List.of("WA", "NSW", "T", "V", "NT", "SA", "Q");
    var values      = List.of(1, 2, 3);  // 1: red, 2: blue, 3: green
    var constraints = List.of(
        "WA != NT", "WA != SA",
        "NT != SA", "NT != Q",
        "SA != Q",  "SA != NSW", "SA != V",
        "Q != NSW", "NSW != V"
    );
    return new CSP(variables, values, constraints);
}

In [ ]:
var csp = mapColoringCSP();
csp

In [ ]:
var solution = solve(csp);
solution

As there are $3$ different values and $7$ different variables, there are $3^7 = 2187$ different ways to assign values to the variables.

Java has no operator for exponentiation.  Instead, we use the method `Math.pow`, which returns a `double`.

In [ ]:
(int) Math.pow(3, 7)

## Visualization

The function `showSolution` takes one parameter:
 * `assignment` is a partial variable assignment, i.e. it is a map
   mapping some of the variables to the numbers representing colors.

It displays a map of Australia where the states are colored according to `assignment`.

In [ ]:
Map<String, String> AUSTRALIA_PATHS = new LinkedHashMap<>();
AUSTRALIA_PATHS.put("WA",  "M 190,340 L 180,310 L 130,280 L 60,250 L 40,200 L 60,150 L 90,120 L 150,80 L 240,70 L 240,250 L 210,340 Z");
AUSTRALIA_PATHS.put("NT",  "M 240,70 L 330,70 L 340,100 L 330,200 L 240,200 Z");
AUSTRALIA_PATHS.put("SA",  "M 240,200 L 330,200 L 330,250 L 360,270 L 350,340 L 290,340 L 260,310 L 210,340 L 240,250 Z");
AUSTRALIA_PATHS.put("Q",   "M 330,70 L 360,20 L 400,60 L 430,150 L 450,200 L 450,250 L 330,250 L 330,200 Z");
AUSTRALIA_PATHS.put("NSW", "M 330,250 L 450,250 L 440,320 L 350,320 L 350,270 Z");
AUSTRALIA_PATHS.put("V",   "M 350,320 L 440,320 L 420,360 L 360,360 Z");
AUSTRALIA_PATHS.put("T",   "M 390,390 L 420,390 L 410,430 L 380,430 Z");

record Label(int x, int y, List<String> lines) {}

Map<String, Label> LABELS = new LinkedHashMap<>();
LABELS.put("WA",  new Label(140, 190, List.of("Western", "Australia")));
LABELS.put("NT",  new Label(285, 130, List.of("Northern", "Territory")));
LABELS.put("SA",  new Label(290, 265, List.of("South", "Australia")));
LABELS.put("Q",   new Label(390, 140, List.of("Queensland")));
LABELS.put("NSW", new Label(395, 280, List.of("New South", "Wales")));
LABELS.put("V",   new Label(395, 340, List.of("Victoria")));
LABELS.put("T",   new Label(400, 415, List.of("Tasmania")));

void showSolution(Map<String, Integer> assignment) {
    Map<String, String> colors = Map.of("red",     "#ff8787",
                                        "green",   "#69db7c",
                                        "blue",    "#4dabf7",
                                        "default", "#f1f3f5");
    var svgPaths = new StringBuilder();
    for (var state : AUSTRALIA_PATHS.keySet()) {
        Integer numVal = assignment.get(state);
        String  fill   = numVal != null ? colors.get(COLOR_MAP.get(numVal)) : colors.get("default");
        svgPaths.append("<path d=\"%s\" fill=\"%s\" stroke=\"#888\" stroke-width=\"1\" />"
                        .formatted(AUSTRALIA_PATHS.get(state), fill));
    }
    var svgLabels = new StringBuilder();
    for (var config : LABELS.values()) {
        var tspans = new StringBuilder();
        for (int i = 0; i < config.lines().size(); i++) {
            tspans.append("<tspan x=\"%d\" dy=\"%s\">%s</tspan>"
                          .formatted(config.x(), i == 0 ? "0" : "1.2em", config.lines().get(i)));
        }
        svgLabels.append("""
            <text x="%d" y="%d" font-family="Arial, sans-serif" font-size="12" font-weight="bold"
                  fill="#343a40" text-anchor="middle" pointer-events="none">%s</text>
            """.formatted(config.x(), config.y(), tspans));
    }
    // IJava treats a line starting with % as a magic command, even inside a text block.
    // Therefore, the placeholders %s must not be at the beginning of a line.
    var html = """
        <div style="border: 1px solid #dee2e6; padding: 10px; border-radius: 5px; display: inline-block; background: white;">
            <svg width="500" height="450" viewBox="0 0 500 450">%s%s</svg>
        </div>
        """.formatted(svgPaths, svgLabels);
    display(html, "text/html");
}

In [ ]:
showSolution(solution);

Below, we have changed the ordering of the variables.  With this new, the constraint solver never needs to backtrack. 

In [ ]:
CSP mapColoringCSP2() {
    var variables   = List.of("WA", "NT", "SA", "Q", "NSW", "V", "T");
    var values      = List.of(1, 2, 3);  // 1: red, 2: blue, 3: green
    var constraints = List.of(
        "WA != NT", "WA != SA",
        "NT != SA", "NT != Q",
        "SA != Q",  "SA != NSW", "SA != V",
        "Q != NSW", "NSW != V"
    );
    return new CSP(variables, values, constraints);
}

In [ ]:
var cspOptimized = mapColoringCSP2();
var optSolution  = solve(cspOptimized);
if (optSolution != null) {
    showSolution(optSolution);
} else {
    System.out.println("No solution found.");
}